# GDN decision fine-tuning on Google Colab

This notebook clones the research repository, fetches the existing 30M scalar/channel-wise checkpoints from the shared Drive archive, smoke-checks and fine-tunes both models, saves and verifies both fine-tuned checkpoints in Google Drive, and only then begins evaluation. It also runs the optional frozen Laya baseline and saves a compressed result bundle to Drive.

**Run the cells in order.** The first cell mounts Drive. Before any large download/training, the notebook checks your Drive quota and Colab disk space and stops if it cannot verify adequate room. Use a Colab GPU runtime (T4 or better); this notebook will not fall back to CPU. Training and LR tuning may take multiple sessions. Rerun an interrupted stage: candidate and model checkpoints are persisted under the Drive run directory for resume.

The repo changes and this notebook must be pushed to the configured GitHub branch before Colab can clone them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Storage and shared-checkpoint preflight

This queries your Google Drive account quota and the shared archive's size, then reserves space for the download, rolling resume checkpoints, model-only checkpoints, and final compressed package. The checks are deliberately conservative and fail closed when Google does not provide quota/file-size metadata. If Drive has less than 5 GiB free or the Colab VM cannot fit the archive plus 5 GiB working space, stop and free space before proceeding.

In [ ]:
from google.colab import auth
auth.authenticate_user()
from google.auth import default
from googleapiclient.discovery import build
from pathlib import Path
import shutil

CHECKPOINT_FILE_ID = '1ThcNNaZssxK1lGjWw9830aU1X4G2vHCl'
DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_DRIVE_DIR = DRIVE_ROOT / 'decision_finetuning_research'
RUN_DIR = PROJECT_DRIVE_DIR / 'run_30M_typed_decisions'
CHECKPOINT_ARCHIVE = Path('/content/shared_checkpoints/results.tar.gz')
LOCAL_CACHE = Path('/content/decision_research_cache')
MIN_DRIVE_FREE_BYTES = 5 * 1024**3
LOCAL_WORKING_RESERVE_BYTES = 5 * 1024**3

credentials, _ = default()
drive_api = build('drive', 'v3', credentials=credentials, cache_discovery=False)
about = drive_api.about().get(fields='storageQuota').execute()
quota = about.get('storageQuota', {})
if not quota.get('limit') or quota.get('usage') is None:
    raise RuntimeError('Google Drive did not report a finite storage quota. Check available Drive space manually; this notebook will not guess.')
drive_free = int(quota['limit']) - int(quota['usage'])
if drive_free < MIN_DRIVE_FREE_BYTES:
    raise RuntimeError(f'Insufficient Google Drive space: {drive_free / 1024**3:.2f} GiB free; at least 5 GiB is required.')
shared_file = drive_api.files().get(fileId=CHECKPOINT_FILE_ID, fields='id,name,size,mimeType', supportsAllDrives=True).execute()
archive_bytes = int(shared_file.get('size', 0))
if archive_bytes <= 0:
    raise RuntimeError('Could not verify the shared results.tar.gz file size through Drive. Confirm link access and retry.')
local_free = shutil.disk_usage('/content').free
required_local = archive_bytes + LOCAL_WORKING_RESERVE_BYTES
print(f"Shared checkpoint archive: {shared_file.get('name')} ({archive_bytes / 1024**3:.2f} GiB)")
print(f'Google Drive free: {drive_free / 1024**3:.2f} GiB; required reserve: 5 GiB')
print(f'Colab local disk free: {local_free / 1024**3:.2f} GiB; required for archive + working space: {required_local / 1024**3:.2f} GiB')
if local_free < required_local:
    raise RuntimeError('Insufficient Colab local disk for the shared archive plus working reserve. Do not start downloading/training.')
PROJECT_DRIVE_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f'Persistent run directory: {RUN_DIR}')

## Confirm a GPU runtime

This checks that Colab actually assigned a CUDA GPU before installing packages or downloading the checkpoints. It performs no model forward pass or smoke training.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is active. In Colab select Runtime > Change runtime type > GPU, reconnect, and rerun from the top.')
print('PyTorch:', torch.__version__)
print('CUDA runtime:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('Compute capability:', torch.cuda.get_device_capability(0))
if torch.cuda.get_device_capability(0) < (7, 5):
    raise RuntimeError('This notebook expects at least a Tesla T4-class GPU (compute capability 7.5).')

## Clone the research code and install runtime dependencies

The install list deliberately does not install/replace PyTorch. Colab's CUDA-enabled PyTorch must remain intact. The repo currently declares a narrow Python version range, so this notebook installs the pipeline's runtime libraries directly rather than using `pip install -e .`. The Laya package is installed later in its own virtual environment.

In [ ]:
import subprocess, sys
REPO_URL = 'https://github.com/code-runner1306/Major-Project-Research.git'
REPO_REF = 'main'
REPO_DIR = Path('/content/Major-Project-Research')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
else:
    remote = subprocess.check_output(['git', '-C', str(REPO_DIR), 'remote', 'get-url', 'origin'], text=True).strip()
    if remote != REPO_URL:
        raise RuntimeError(f'Existing path is not the expected repository: {remote}')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_REF], check=True)
PIPELINE = REPO_DIR / 'decision_research.py'
if not PIPELINE.is_file() or not (REPO_DIR / 'decision_baseline.py').is_file():
    raise RuntimeError('The cloned branch does not contain the decision-research implementation. Commit/push the code changes first.')
COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
print('Repository commit:', COMMIT)
packages = ['datasets==4.6.1', 'tiktoken==0.12.0', 'transformers==4.57.6', 'scipy==1.17.1', 'matplotlib==3.10.8', 'tabulate==0.10.0', 'gdown', 'huggingface_hub', 'google-api-python-client', 'nvidia-ml-py==13.595.45']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
help_text = subprocess.check_output([sys.executable, str(PIPELINE), '--help'], cwd=REPO_DIR, text=True)
for command in ['prepare', 'tune', 'train', 'calibrate', 'evaluate', 'baseline', 'report', 'package']:
    if command not in help_text:
        raise RuntimeError(f'Cloned pipeline is missing the {command} command.')
print('Pipeline CLI and dependencies are ready; PyTorch was not reinstalled.')

## Download and verify the original pretrained checkpoints

The archive is downloaded to Colab's local disk, not copied into Drive again. The run itself and its recoverable checkpoints live under `RUN_DIR` on mounted Drive. This verifies the expected scalar seed 43 and channel-wise seed 42 archive members before training.

In [ ]:
import gdown, tarfile, hashlib
CHECKPOINT_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
if not CHECKPOINT_ARCHIVE.exists():
    downloaded = gdown.download(id=CHECKPOINT_FILE_ID, output=str(CHECKPOINT_ARCHIVE), quiet=False)
    if not downloaded:
        raise RuntimeError('Could not download the shared checkpoint archive.')
expected_members = {
    'scalar': 'tinystories/30M/scalar/seed_43_ef7dde5a783e/final_checkpoint.pt',
    'channel-wise': 'tinystories/30M/channel-wise/seed_42_ced68e206c8b/final_checkpoint.pt',
}
with tarfile.open(CHECKPOINT_ARCHIVE, 'r:gz') as archive:
    names = [member.name.replace('\\', '/').lstrip('./') for member in archive.getmembers() if member.isfile()]
    for variant, expected in expected_members.items():
        matches = [name for name in names if name.endswith(expected)]
        if len(matches) != 1:
            raise RuntimeError(f'Expected exactly one {variant} checkpoint ending in {expected}; found {matches}')
archive_digest = hashlib.sha256()
with CHECKPOINT_ARCHIVE.open('rb') as archive_stream:
    for chunk in iter(lambda: archive_stream.read(1024 * 1024), b''):
        archive_digest.update(chunk)
archive_hash = archive_digest.hexdigest()
print(f'Archive verified: {CHECKPOINT_ARCHIVE.stat().st_size / 1024**3:.2f} GiB; SHA-256 {archive_hash}')
print('Both selected pretrained checkpoints are present.')

## Run commands with persistent logs

Each stage streams output to the notebook and writes a log under the Drive run directory. You can inspect those logs after reconnecting. A stage that was interrupted can be rerun; tuning candidates and production fine-tunes use rolling checkpoints in Drive.

In [ ]:
import os
LOCAL_CACHE.mkdir(parents=True, exist_ok=True)
existing_manifest_path = RUN_DIR / 'run_manifest.json'
DATASET_REVISION = json.loads(existing_manifest_path.read_text()).get('dataset_revision', 'main') if existing_manifest_path.exists() else 'main'
COMMON_ARGS = ['--run-dir', str(RUN_DIR), '--output-root-local', str(LOCAL_CACHE), '--dataset-revision', DATASET_REVISION]
def run_stage(command, log_name, common_args=None):
    log_dir = RUN_DIR / 'notebook_logs'
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / log_name
    full_command = [sys.executable, str(PIPELINE), *(COMMON_ARGS if common_args is None else common_args), *command]
    print('Running:', ' '.join(full_command))
    with log_path.open('a', encoding='utf-8') as log:
        proc = subprocess.Popen(full_command, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in proc.stdout:
                print(line, end='')
                log.write(line); log.flush()
        except KeyboardInterrupt:
            print(f'Interrupted. Check Drive checkpoints/logs before rerunning. Log: {log_path}')
            proc.terminate()
            try:
                proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                proc.kill(); proc.wait()
            raise
        code = proc.wait()
    if code != 0:
        raise subprocess.CalledProcessError(code, full_command)
    print('Stage complete. Log:', log_path)

def archive_args():
    return ['--checkpoint-archive', str(CHECKPOINT_ARCHIVE)]

## Prepare data and smoke-check both fine-tuning paths

`prepare` resolves and records the exact Typed Decisions dataset revision, creates case-isolated train/validation/calibration splits, and checks the context-length gate. The next cell runs the repository's short GPU preflight separately for scalar and channel-wise models using only small train/validation subsets. It checks that both model paths can fine-tune and emit checkpoint artifacts before spending time on LR tuning and production. This does not read the official test split or replace the formal tests.

In [ ]:
run_stage(['prepare'], 'prepare.log')


In [ ]:
for variant in ['scalar', 'channel-wise']:
    run_stage(['preflight', '--variant', variant, *archive_args()], f'smoke_{variant}.log')
    smoke_dir = RUN_DIR / 'preflight' / variant
    expected = [smoke_dir / 'best_model.pt', smoke_dir / 'final_model.pt']
    missing = [str(path) for path in expected if not path.is_file() or path.stat().st_size == 0]
    if missing:
        raise RuntimeError(f'{variant} preflight did not produce valid best/final checkpoint files: {missing}')
    from decision_model import GDNDecisionModel
    reloaded = GDNDecisionModel.load_inference_checkpoint(expected[0], map_location='cpu')
    if reloaded.variant != variant: raise RuntimeError(f'Reloaded smoke checkpoint variant mismatch: {reloaded.variant}')
    from decision_research import _trainer_rows
    from decision_evaluation import predict_gdn, fit_primitive_temperatures, apply_temperatures, summarize_predictions
    import math
    calibration_examples = _trainer_rows(RUN_DIR / 'data' / 'calibration.jsonl')
    smoke_examples = [next(row for row in calibration_examples if row['primitive'] == primitive) for primitive in ('choice', 'score', 'noul')]
    reloaded = reloaded.to('cuda')
    smoke_raw = predict_gdn(reloaded, smoke_examples, device=torch.device('cuda'), partition='calibration')
    smoke_temperatures = fit_primitive_temperatures(smoke_raw, max_iter=10)
    smoke_calibrated = apply_temperatures(smoke_raw, smoke_temperatures)
    smoke_metrics = summarize_predictions(smoke_calibrated, probability_key='calibrated_probabilities')
    if smoke_metrics['examples'] != 3 or not all(math.isfinite(v) for v in smoke_metrics['overall'].values() if isinstance(v, (int, float))):
        raise RuntimeError(f'{variant} prediction/calibration/metrics smoke check failed: {smoke_metrics}')
    del reloaded
    print(f'{variant} smoke passed: train → save → reload → choice/score/noul prediction → calibration → metrics; artifacts:', *(str(p) for p in expected))
print('Both variants passed the bounded pipeline smoke on development data. Official-test evaluation has not started.')

## Select and lock the shared learning rate

The bounded comparison uses validation data only and evaluates both candidate learning rates on both variants. It may take a while on a free Colab GPU. The official test split remains untouched.

In [ ]:
tuning_result_path = RUN_DIR / 'tuning' / 'learning_rate_results.json'
if not tuning_result_path.exists():
    run_stage(['tune', *archive_args()], 'tune.log')
tuning = json.loads(tuning_result_path.read_text())
SELECTED_LR = float(tuning['selected_backbone_learning_rate'])
print('Locked shared backbone learning rate:', SELECTED_LR)

## Fine-tune the two selected 30M checkpoints

The models run sequentially to reduce peak GPU memory. If Colab disconnects, reconnect, rerun setup cells, then rerun this cell: each variant restores its latest compatible optimizer-boundary checkpoint and skips completed work. Do not start both variants concurrently.

In [ ]:
for variant in ['scalar', 'channel-wise']:
    run_stage(['train', '--variant', variant, '--learning-rate', str(SELECTED_LR), *archive_args()], f'train_{variant}.log')

## Archive and verify all fine-tuned GDN weights in Drive — mandatory before evaluation

This barrier runs only after the cell above has completed training for both variants. It copies each variant's model-only `best_model.pt` and `final_model.pt` into a single Drive ZIP, records SHA-256 hashes and sizes, then reopens the Drive copy and verifies it. Do not start calibration or evaluation unless this cell completes successfully. The original resumable optimizer checkpoints remain in `RUN_DIR`; this archive is a durable, portable copy of all four model-only fine-tuned exports.

In [ ]:
import hashlib, zipfile
from datetime import datetime, timezone
WEIGHTS_ARCHIVE = PROJECT_DRIVE_DIR / 'gdn_finetuned_weights_before_evaluation.zip'
weight_files = {f'{variant}/{name}.pt': RUN_DIR / variant / 'checkpoints' / f'{name}.pt'
                for variant in ['scalar', 'channel-wise']
                for name in ['best_model', 'final_model']}
missing = [str(path) for path in weight_files.values() if not path.is_file() or path.stat().st_size == 0]
if missing:
    raise RuntimeError(f'Both production fine-tunes must finish and export best/final weights first. Missing: {missing}')
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
weight_manifest = {'schema_version': 1, 'created_utc': datetime.now(timezone.utc).isoformat(), 'repository_commit': COMMIT, 'evaluation_started': False, 'files': {name: {'bytes': path.stat().st_size, 'sha256': sha256_file(path)} for name, path in weight_files.items()}}
estimated_zip_bytes = sum(entry['bytes'] for entry in weight_manifest['files'].values()) + 1024 * 1024
about = drive_api.about().get(fields='storageQuota').execute()['storageQuota']
drive_free = int(about['limit']) - int(about['usage'])
if drive_free < estimated_zip_bytes + 256 * 1024**2:
    raise RuntimeError(f'Not enough free Drive space for the pre-evaluation weights archive plus safety reserve: need about {estimated_zip_bytes / 1024**3:.2f} GiB, have {drive_free / 1024**3:.2f} GiB.')
local_zip = Path('/content/gdn_finetuned_weights_before_evaluation.zip')
with zipfile.ZipFile(local_zip, 'w', compression=zipfile.ZIP_STORED) as archive:
    for arcname, source in weight_files.items(): archive.write(source, arcname)
    archive.writestr('weights_manifest.json', json.dumps(weight_manifest, indent=2))
temporary_drive_zip = WEIGHTS_ARCHIVE.with_suffix('.zip.partial')
shutil.copy2(local_zip, temporary_drive_zip)
if temporary_drive_zip.stat().st_size != local_zip.stat().st_size:
    raise IOError('Drive copy size differs from the locally created checkpoint archive.')
with zipfile.ZipFile(temporary_drive_zip, 'r') as archive:
    bad_member = archive.testzip()
    if bad_member is not None: raise IOError(f'Corrupt ZIP member: {bad_member}')
    saved_manifest = json.loads(archive.read('weights_manifest.json'))
    for name, entry in saved_manifest['files'].items():
        info = archive.getinfo(name)
        if info.file_size != entry['bytes']: raise IOError(f'Checkpoint size verification failed: {name}')
        with archive.open(name) as member:
            digest = hashlib.sha256()
            for chunk in iter(lambda: member.read(1024 * 1024), b''): digest.update(chunk)
        if digest.hexdigest() != entry['sha256']: raise IOError(f'Checkpoint SHA-256 verification failed: {name}')
os.replace(temporary_drive_zip, WEIGHTS_ARCHIVE)
if not WEIGHTS_ARCHIVE.is_file(): raise RuntimeError('Pre-evaluation weights archive was not committed to Drive.')
print('Verified and saved all four fine-tuned checkpoints to Drive:', WEIGHTS_ARCHIVE)
print(f'Archive size: {WEIGHTS_ARCHIVE.stat().st_size / 1024**3:.2f} GiB; SHA-256: {sha256_file(WEIGHTS_ARCHIVE)}')
print('Evaluation is still not run at this point; proceed to the next cell only after this one succeeds.')

## Calibrate and evaluate the GDN models

This is the first evaluation section. Its prerequisite cell above must have created and SHA-256-verified the Drive checkpoint archive after both scalar and channel-wise fine-tunes completed. Calibration uses only the held-out calibration partition; `evaluate` is the first stage that loads the official test split and also runs canonical and deterministic option-order evaluations. The frozen Laya baseline is an inference-only comparator, not a fine-tuned model, and is run in its separate section below. Do not alter training settings after the locked profile is created.

In [ ]:
if not WEIGHTS_ARCHIVE.is_file() or not zipfile.is_zipfile(WEIGHTS_ARCHIVE):
    raise RuntimeError('Refusing to begin evaluation: verified pre-evaluation checkpoint archive is missing.')
for variant in ['scalar', 'channel-wise']:
    run_stage(['calibrate', '--variant', variant], f'calibrate_{variant}.log')
    run_stage(['evaluate', '--variant', variant], f'evaluate_{variant}.log')

## Laya baseline in an isolated environment

Laya's current package release is installed in a separate virtual environment so it cannot replace Colab's CUDA PyTorch or the GDN runtime dependencies. This adds a separate model download (the typed-decision checkpoint is about 421M parameters). The adapter pins and records the resolved Hugging Face model commit. This stage is needed for the complete comparison and for the final completeness-checked package; it can be deferred, but do not run the final `report`/`package` cells before it completes.

In [ ]:
import venv
LAYA_ENV = Path('/content/laya-decision-env')
if not LAYA_ENV.exists():
    venv.EnvBuilder(with_pip=True, system_site_packages=True).create(LAYA_ENV)
LAYA_PYTHON = LAYA_ENV / 'bin' / 'python'
subprocess.run([str(LAYA_PYTHON), '-m', 'pip', 'install', '-q', 'laya==0.3.21'], check=True)
subprocess.run([str(LAYA_PYTHON), '-c', "import laya, importlib.metadata; print('Laya', importlib.metadata.version('laya'))"], check=True)
laya_command = [str(LAYA_PYTHON), str(PIPELINE), *COMMON_ARGS, 'baseline']
# Stream the optional baseline log with the same durable run directory.
with (RUN_DIR / 'notebook_logs' / 'baseline_laya.log').open('a', encoding='utf-8') as log:
    proc = subprocess.Popen(laya_command, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end=''); log.write(line); log.flush()
    if proc.wait() != 0:
        raise RuntimeError('Laya baseline stage failed; see baseline_laya.log')

## Generate reports and create the final Drive archive

Packaging refuses to proceed unless both GDN models, calibration/test predictions, Laya predictions, and a report are present. The archive also includes `telemetry/evaluation_energy.json` (inference energy, runtime/device identity, GPU-activity samples, and measurement conditions), `telemetry/training_resources.json` (per-session and cumulative fine-tuning time, optimizer steps, energy, and device activity), and grouped latency summaries by primitive, serialized-token-length bucket, and option count. Unsupported telemetry is retained as unavailable rather than guessed. It excludes source pretraining checkpoints and rolling resume slots. It is stored in the Drive run folder and copied to a clearly named location in your Drive root. The earlier quota check reserves room for this duplicate package copy.

In [ ]:
run_stage(['report'], 'report.log')
run_stage(['package'], 'package.log')
import shutil, hashlib
source_archive = RUN_DIR / 'decision_research_results.tar.gz'
final_archive = PROJECT_DRIVE_DIR / 'decision_research_results_30M.tar.gz'
shutil.copy2(source_archive, final_archive)
digest_obj = hashlib.sha256()
with final_archive.open('rb') as archive_stream:
    for chunk in iter(lambda: archive_stream.read(1024 * 1024), b''):
        digest_obj.update(chunk)
digest = digest_obj.hexdigest()
print('Final archive:', final_archive)
print(f'Size: {final_archive.stat().st_size / 1024**3:.2f} GiB')
print('SHA-256:', digest)
print('Drive run directory:', RUN_DIR)
print('Reports:', RUN_DIR / 'reports')
print('Training costs and activity:', RUN_DIR / 'telemetry' / 'training_resources.json')
print('Evaluation energy/runtime/activity:', RUN_DIR / 'telemetry' / 'evaluation_energy.json')
print('Latency groups:', RUN_DIR / 'reports' / 'latency_summary.csv')